In [2]:
# Install libraries
!pip install --quiet transformers evaluate jiwer librosa soundfile matplotlib seaborn pandas tqdm openai-whisper

In [3]:
# Mount drive
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Import preprocessing and normalization functions
import sys
sys.path.insert(0, '/content/drive/MyDrive/Target TTS Project/TargetTTS')

from src.preprocess2 import preprocess_audio, normalize_text, build_waxal_cache, TARGET_SAMPLE_RATE

In [7]:
# Import libraries
import os
import re
import torch
import jiwer
import librosa
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from tqdm import tqdm
from datasets import load_from_disk, Audio as HFAudio
from transformers import WhisperProcessor, WhisperForConditionalGeneration

In [38]:
# Base Path
BASE_PATH                = "/content/drive/MyDrive/Target TTS Project/TargetTTS"

# English (w/ Base Whisper)
ENG_CSV                  = f"{BASE_PATH}/data/mixture_recipes/librispeech_mixture_recipes.csv"
ENG_AUDIO                = f"{BASE_PATH}/data/synthetic_mixtures/librispeech_synthetic_mixtures"
ENG_BENCHMARK_OUTPUT_CSV = f"{BASE_PATH}/data/benchmark_results/librispeech_benchmark_results.csv"
ENG_MODEL                = "openai/whisper-small" # English model

# Twi (w/ Fine-Tuned Whisper)
TWI_CSV                  = f"{BASE_PATH}/data/mixture_recipes/waxal_mixture_recipes.csv"
TWI_AUDIO                = f"{BASE_PATH}/data/synthetic_mixtures/waxal_synthetic_mixtures"
TWI_BENCHMARK_OUTPUT_CSV = f"{BASE_PATH}/data/benchmark_results/waxal_benchmark_results.csv"
TWI_MODEL                = f"{BASE_PATH}/models/whisper-small-twi/final" # Finetuned Twi model
WAXAL_DISK_PATH          = f"{BASE_PATH}/data/waxal"

# Plots Output Path
PLOTS_OUTPUT_PATH        = f"{BASE_PATH}/data/benchmark_results/plots"

## Adding Baseline Rows

In [39]:
def add_baseline_rows(input_csv, output_csv, num_samples=500):
    """Appends baseline (clean, unmixed) audio rows to an existing mix recipe CSV."""
    print(f"Processing {input_csv}...")
    df = pd.read_csv(input_csv)

    if df['overlap_ratio'].eq(0.0).any():
        print(f"Baseline rows already present in {input_csv}, skipping.")
        return

    # Grab unique target files to avoid re-testing the same audio
    unique_targets = df.drop_duplicates(subset=['target_audio_path']).head(num_samples).copy()

    if len(unique_targets) < num_samples:
      print(f"    Only {len(unique_targets)} unique targets available, fewer than requested {num_samples}.")

    # Override the parameters to turn these into 0.0 overlap baselines
    unique_targets['overlap_ratio'] = 0.0
    unique_targets['sir_level_db'] = 0  # Placeholder since SIR doesn't matter for clean audio
    unique_targets['noise_audio_path'] = "NONE"
    unique_targets['noise_transcript'] = "" # Baseline audio so 0 leakage
    unique_targets['mix_id'] = unique_targets['mix_id'].apply(lambda x: x + "_baseline")
    unique_targets['noise_start_times'] = "[]"

    # Append the new baseline rows to the original dataframe
    df_updated = pd.concat([df, unique_targets], ignore_index=True)

    df_updated.to_csv(output_csv, index=False)
    print(f"Added {len(unique_targets)} baseline (0.0 overlap) rows. Saved to {output_csv}\n")

In [40]:
# Overwrites the original files
add_baseline_rows(ENG_CSV, ENG_CSV)
add_baseline_rows(TWI_CSV, TWI_CSV)

Processing /content/drive/MyDrive/Target TTS Project/TargetTTS/data/mixture_recipes/librispeech_mixture_recipes.csv...
Baseline rows already present in /content/drive/MyDrive/Target TTS Project/TargetTTS/data/mixture_recipes/librispeech_mixture_recipes.csv, skipping.
Processing /content/drive/MyDrive/Target TTS Project/TargetTTS/data/mixture_recipes/waxal_mixture_recipes.csv...
Baseline rows already present in /content/drive/MyDrive/Target TTS Project/TargetTTS/data/mixture_recipes/waxal_mixture_recipes.csv, skipping.


## Function Definitions

In [42]:
def calculate_leakage(pred, target, noise):
    """Calculates percentage of noise words that leaked into the prediction."""
    pred_words = set(pred.split())
    target_words = set(target.split())
    noise_words = set(noise.split())

    # Leaked words: Words the model predicted that are in the noise, BUT NOT in the target
    leaked_words = (pred_words & noise_words) - target_words

    if len(noise_words) == 0:
        return 0.0
    return len(leaked_words) / len(noise_words)

In [43]:
# Model Initialization
def load_whisper_model(model_id, device):
    """Initializes the Whisper model and processor based on model_id."""
    print(f"Loading model: {model_id}...")

    is_local = os.path.isdir(model_id)

    processor = WhisperProcessor.from_pretrained(model_id, local_files_only=is_local)
    model = WhisperForConditionalGeneration.from_pretrained(model_id, local_files_only=is_local).to(device)
    model.eval()

    return processor, model

In [44]:
# Checkpoint Management
def load_checkpoint(output_csv_path):
    """Loads existing progress to resume from a checkpoint."""
    if os.path.exists(output_csv_path):
        df_done = pd.read_csv(output_csv_path)
        done_ids = set(df_done['mix_id'].tolist())
        results = df_done.to_dict('records')
        print(f"Resuming from checkpoint: {len(done_ids)} already done.")
        return done_ids, results
    return set(), []

In [45]:
# Audio Fetching
def fetch_audio_array(row, audio_dir, audio_cache, base_path):
    """Determines the correct audio path, loads, and preprocesses the audio."""
    mix_id = row['mix_id']
    sir = row['sir_level_db']
    overlap = row['overlap_ratio']

    # Handle 0.0 Overlap (Clean Baseline)
    if overlap == 0.0:
        if audio_cache is not None:
            key = row['target_audio_path']
            if key not in audio_cache:
                print(f"Baseline audio not found in cache: {key}")
                return None
            return audio_cache[key].astype(np.float32)
        else:
            p = row['target_audio_path']
            audio_path = p if os.path.isabs(p) else os.path.join(base_path, p)

    # Handle Mixed Audio
    else:
        audio_path = os.path.join(audio_dir, f"{mix_id}_sir{int(sir)}_ov{round(overlap, 1)}.wav")

    # Final Check & Load
    if not os.path.exists(audio_path):
        print(f"Audio not found: {audio_path}")
        return None

    waveform = preprocess_audio(audio_path)
    return waveform.squeeze(0).numpy()

In [46]:
# Inference and Metrics
def transcribe_and_evaluate(audio_array, target_text, noise_text, model, processor, device, target_sr=16000):
    """Runs Whisper inference and calculates WER, CER, and Leakage."""
    inputs = processor(audio_array, sampling_rate=target_sr, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    forced_decoder_ids = processor.get_decoder_prompt_ids(language="english", task="transcribe") # The twi model also requires langugae="english"

    with torch.no_grad():
        predicted_ids = model.generate(inputs["input_features"], forced_decoder_ids=forced_decoder_ids)

    pred_text = normalize_text(processor.batch_decode(predicted_ids, skip_special_tokens=True)[0])

    wer = jiwer.wer(target_text, pred_text)
    cer = jiwer.cer(target_text, pred_text)
    leakage = calculate_leakage(pred_text, target_text, noise_text)

    return pred_text, wer, cer, leakage

In [47]:
def run_benchmark(csv_path, audio_dir, model_id, output_csv_path, audio_cache=None):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    BASE_PATH = "/content/drive/MyDrive/Target TTS Project/TargetTTS"

    # Setup Data and Checkpoints
    df = pd.read_csv(csv_path)
    os.makedirs(os.path.dirname(output_csv_path), exist_ok=True)
    done_ids, results = load_checkpoint(output_csv_path)

    # Setup Model
    processor, model = load_whisper_model(model_id, device)

    # Main Loop
    print(f"Starting evaluation for {len(df)} mixes...")

    for index, row in tqdm(df.iterrows(), total=df.shape[0]):
        if row['mix_id'] in done_ids:
            continue

        target_text = normalize_text(str(row['target_transcript']) if pd.notna(row['target_transcript']) else "")
        noise_text = normalize_text(str(row['noise_transcript']) if pd.notna(row['noise_transcript']) else "")

        # Skip empty ground truths
        if len(target_text) == 0:
            continue

        # Get audio
        audio_array = fetch_audio_array(row, audio_dir, audio_cache, BASE_PATH)
        if audio_array is None:
            continue

        # Inference & Scoring
        pred_text, wer, cer, leakage = transcribe_and_evaluate(
            audio_array, target_text, noise_text, model, processor, device, TARGET_SAMPLE_RATE
        )

        # Log Results
        row_result = row.to_dict()
        row_result.update({'pred_transcript': pred_text, 'wer': wer, 'cer': cer, 'leakage_rate': leakage})
        results.append(row_result)

        # Periodic Saving (every 100 iterations)
        if len(results) % 100 == 0:
            pd.DataFrame(results).to_csv(output_csv_path, index=False)

    # Final Save
    df_results = pd.DataFrame(results)
    df_results.to_csv(output_csv_path, index=False)
    print(f"Benchmark complete. Saved to {output_csv_path}")

    return df_results

In [48]:
def analyze_and_plot(benchmnark_results_csv, dataset_name, output_dir="plots"):
    os.makedirs(output_dir, exist_ok=True)
    df = pd.read_csv(benchmnark_results_csv)

    # Aggregation per Bucket
    print(f"\n=== {dataset_name} Performance by Bucket ===")
    bucket_stats = df.groupby(['sir_level_db', 'overlap_ratio'])[['wer', 'cer', 'leakage_rate']].mean().reset_index()
    print(bucket_stats.to_string(index=False))

    # Save the raw aggregated table
    bucket_stats.to_csv(f"{output_dir}/{dataset_name}_aggregated_stats.csv", index=False)

    # Plotting Setup
    sns.set_theme(style="whitegrid")

    # Plot A: WER vs. SIR Level (Grouped by Overlap)
    # Excluding 0.0 overlap here because SIR doesn't apply to clean audio
    df_overlap_only = df[df['overlap_ratio'] > 0.0]

    plt.figure(figsize=(10, 6))
    ax = sns.lineplot(data=df_overlap_only, x='sir_level_db', y='wer', hue='overlap_ratio', marker='o', palette="viridis")
    plt.title(f"{dataset_name}: Word Error Rate (WER) vs. Signal-to-Interference Ratio (SIR)", fontsize=14)
    plt.xlabel("SIR Level (dB)", fontsize=12)
    plt.ylabel("Average WER", fontsize=12)
    plt.legend(title='Overlap Ratio')
    plt.savefig(f"{output_dir}/{dataset_name}_WER_vs_SIR.png", dpi=300, bbox_inches='tight')
    plt.close()

    # Plot B: WER vs. Overlap Ratio (Grouped by SIR)
    # Including the 0.0 baseline here
    plt.figure(figsize=(10, 6))
    sns.lineplot(data=df, x='overlap_ratio', y='wer', hue='sir_level_db', marker='s', palette="magma")
    plt.title(f"{dataset_name}: Word Error Rate (WER) vs. Overlap Ratio", fontsize=14)
    plt.xlabel("Overlap Ratio", fontsize=12)
    plt.ylabel("Average WER", fontsize=12)
    plt.legend(title='SIR Level (dB)')
    plt.savefig(f"{output_dir}/{dataset_name}_WER_vs_Overlap.png", dpi=300, bbox_inches='tight')
    plt.close()

    # Plot C: Non-Target Leakage vs Overlap
    plt.figure(figsize=(10, 6))
    sns.barplot(data=df_overlap_only, x='overlap_ratio', y='leakage_rate', hue='sir_level_db', palette="Blues_d")
    plt.title(f"{dataset_name}: Non-Target Leakage Rate vs. Overlap Ratio", fontsize=14)
    plt.xlabel("Overlap Ratio", fontsize=12)
    plt.ylabel("Leakage Rate (Percentage of Noise Words in Output)", fontsize=12)
    plt.legend(title='SIR Level (dB)')
    plt.savefig(f"{output_dir}/{dataset_name}_Leakage_vs_Overlap.png", dpi=300, bbox_inches='tight')
    plt.close()

    print(f"Generated 3 plots in '{output_dir}/'")

## Benchmarking

### English Benchmarking

In [ ]:
# Benchmark mixed English audios
df_eng_results = run_benchmark(ENG_CSV, ENG_AUDIO, ENG_MODEL, output_csv_path=ENG_BENCHMARK_OUTPUT_CSV)

### Twi Benchmarking

In [49]:
# Build in-memory cache of WAXAL data
waxal_audio_cache = build_waxal_cache(WAXAL_DISK_PATH)

Loading WAXAL dataset from /content/drive/MyDrive/Datasets/waxalnlp_aka_asr_disk...


Loading dataset from disk:   0%|          | 0/109 [00:00<?, ?it/s]

Building cache: 100%|██████████| 1522/1522 [00:25<00:00, 59.70it/s]

Cached 1522 audio arrays.


In [ ]:
# Benchmark mixed Twi audios
df_twi_results = run_benchmark(TWI_CSV, TWI_AUDIO, TWI_MODEL, output_csv_path=TWI_BENCHMARK_OUTPUT_CSV, audio_cache=waxal_audio_cache)

Loading model: /content/drive/MyDrive/Target TTS Project/TargetTTS/models/whisper-small-twi/final...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Starting evaluation for 13000 mixes...


 16%|█▌        | 2108/13000 [33:04<2:37:05,  1.16it/s]

## Plotting

In [ ]:
# Create and save English benchmark results plots
analyze_and_plot(benchmnark_results_csv=ENG_BENCHMARK_OUTPUT_CSV, dataset_name="English_Base_Whisper", output_dir=PLOTS_OUTPUT_PATH)

In [ ]:
# Create and save Twi benchmark results plots
analyze_and_plot(results_csv=TWI_BENCHMARK_OUTPUT_CSV, dataset_name="Twi_Finetuned_Whisper", output_dir=PLOTS_OUTPUT_PATH)